In [3]:
import os, glob, random, math
from dataclasses import dataclass
from typing import Dict, List, Tuple
from tqdm import tqdm

import numpy as np
from PIL import Image
from PIL import ImageFile

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from transformers import SegformerForSemanticSegmentation, SegformerImageProcessor

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)


DATA_ROOT = "./datasets"


CITIES = ["tunis", "manila", "copenhagen"]
YEARS_TRAIN_SEQ = [2014, 2020, 2025]  

# Your fine-tuned SegFormer checkpoint (HF folder or model id)
SEGFORMER_CKPT = "./models/segformer_osm_esri_grouped/final"


NUM_CLASSES = 7

# Tile size expected (your images should be consistent)
IMG_SIZE = 256   # 256 recommended (faster). You can use 512 if GPU allows.

# Training
BATCH_SIZE = 8
EPOCHS = 25
LR = 2e-4
SEED = 42

# Avoid decompression bomb errors
Image.MAX_IMAGE_PIXELS = None
ImageFile.LOAD_TRUNCATED_IMAGES = True

def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

seed_everything(SEED)


Device: cuda


In [4]:
processor = SegformerImageProcessor.from_pretrained("nvidia/segformer-b0-finetuned-ade-512-512")
segformer = SegformerForSemanticSegmentation.from_pretrained(SEGFORMER_CKPT).to(device)
segformer.eval()

# Freeze SegFormer (we only train the temporal model)
for p in segformer.parameters():
    p.requires_grad = False

@torch.no_grad()
def segformer_predict_mask(pil_img: Image.Image) -> torch.Tensor:
    """
    Returns predicted class-id mask: (H, W) torch.long on CPU
    """
    # Ensure consistent size
    pil_img = pil_img.resize((IMG_SIZE, IMG_SIZE), resample=Image.BILINEAR)

    inputs = processor(images=pil_img, return_tensors="pt")
    inputs = {k: v.to(device) for k, v in inputs.items()}

    outputs = segformer(**inputs)
    logits = outputs.logits  # (B, C, h, w) usually smaller than IMG_SIZE

    # Upsample to IMG_SIZE
    logits_up = F.interpolate(logits, size=(IMG_SIZE, IMG_SIZE), mode="bilinear", align_corners=False)
    pred = torch.argmax(logits_up, dim=1).squeeze(0).to("cpu").long()  # (H, W)

    return pred


c:\Users\2640870\AppData\Local\anaconda3\envs\torch_gpu\lib\site-packages\transformers\image_processing_base.py:417: UserWarning: The following named arguments are not valid for `SegformerImageProcessor.__init__` and were ignored: 'feature_extractor_type', 'reduce_labels'
  image_processor = cls(**image_processor_dict)


In [5]:
import os, glob, random
from typing import Dict, List, Tuple
from tqdm import tqdm

# Root folder for your data (set this somewhere above)
# DATA_ROOT = "/path/to/DATA_ROOT"
# CITIES = ["tunis", "copenhagen", "manila", ...]

IMG_EXTS = (".jpeg")  # currently only matching .jpeg files


def list_tiles_by_folder(city: str, year: int) -> Dict[str, str]:
    """
    Returns dict: tile_id -> filepath

    Expected structure:
      DATA_ROOT/<city>/<tile_id>/tiles/*<year>*.jpeg
    Example:
      tunis/out_12345/ESRI 2025.jpeg

    Note: tile_id can be any folder name, not just starting with 'out_'.
    """
    # search inside each <tile_id>/ folder for an image containing the year in its name
    # changed from "out_*" to "*" so all subfolders are considered
    pattern = os.path.join(DATA_ROOT, city, "tiles" ,"*", "*")
    files = [f for f in glob.glob(pattern) if f.lower().endswith(IMG_EXTS)]

    out: Dict[str, str] = {}
    year_str = str(year)

    for f in files:
        base = os.path.basename(f)
        if year_str not in base:
            continue

        tile_id = os.path.basename(os.path.dirname(f))  # e.g., out_12345, tile_1, etc.

        # If multiple matches exist, keep the first one; or override deterministically
        # Here: prefer "ESRI" if present, otherwise keep existing.
        if tile_id not in out:
            out[tile_id] = f
        else:
            if "esri" in base.lower() and "esri" not in os.path.basename(out[tile_id]).lower():
                out[tile_id] = f

    return out


def build_triplets() -> List[Tuple[str, str, str, str]]:
    """
    Returns list of triplets: (path_2014, path_2020, path_2025, tile_id)
    Only keeps tiles available in all 3 years.
    """
    triplets: List[Tuple[str, str, str, str]] = []

    for city in tqdm(CITIES, desc="Processing cities"):
        print(f"Processing city: {city}")
        m2014 = list_tiles_by_folder(city, 2014)
        m2020 = list_tiles_by_folder(city, 2020)
        m2025 = list_tiles_by_folder(city, 2025)

        common = sorted(set(m2014) & set(m2020) & set(m2025))
        print(city, "common tiles:", len(common))

        for tid in common:
            triplets.append((m2014[tid], m2020[tid], m2025[tid], tid))

    random.shuffle(triplets)
    return triplets


triplets = build_triplets()
print("Total triplets:", len(triplets))

split = int(0.9 * len(triplets))
train_triplets = triplets[:split]
val_triplets = triplets[split:]
print("Train:", len(train_triplets), "Val:", len(val_triplets))


Processing cities:   0%|          | 0/3 [00:00<?, ?it/s]

Processing city: tunis


Processing cities:  33%|███▎      | 1/3 [00:36<01:13, 36.97s/it]

tunis common tiles: 59143
Processing city: manila


Processing cities:  67%|██████▋   | 2/3 [01:27<00:45, 45.09s/it]

manila common tiles: 59143
Processing city: copenhagen


Processing cities: 100%|██████████| 3/3 [01:53<00:00, 37.79s/it]

copenhagen common tiles: 59143
Total triplets: 177429
Train: 159686 Val: 17743


In [6]:
from typing import Tuple
from tqdm.auto import tqdm
from PIL import Image
import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

Image.MAX_IMAGE_PIXELS = None

def resize_image(image: Image.Image, size: Tuple[int, int] = (256, 256)) -> Image.Image:
    return image.resize(size, resample=Image.BILINEAR)


class MaskSequenceDataset(Dataset):
    """
    Returns:
      x_seq: (T=2, C, H, W) one-hot masks for [2014, 2020]
      y:     (H, W) class-id mask for 2025
    """
    def __init__(self, triplets, num_classes: int):
        self.triplets = triplets
        self.num_classes = num_classes

    def __len__(self):
        return len(self.triplets)

    def __getitem__(self, idx):
        p2014, p2020, p2025, tid = self.triplets[idx]

        try:
            # --- load + resize ---
            img2014 = resize_image(Image.open(p2014).convert("RGB"))
            img2020 = resize_image(Image.open(p2020).convert("RGB"))
            img2025 = resize_image(Image.open(p2025).convert("RGB"))

            # --- run segformer (your function) ---
            m2014 = segformer_predict_mask(img2014)  # (H, W) long
            m2020 = segformer_predict_mask(img2020)
            m2025 = segformer_predict_mask(img2025)

            # --- one-hot encode inputs ---
            oh2014 = F.one_hot(m2014, num_classes=self.num_classes).permute(2, 0, 1).float()
            oh2020 = F.one_hot(m2020, num_classes=self.num_classes).permute(2, 0, 1).float()

            x_seq = torch.stack([oh2014, oh2020], dim=0)  # (T=2, C, H, W)
            y = m2025  # (H,W) long

            return x_seq, y

        except Exception as e:
            print(f"[SKIP] tile {tid} at idx {idx}: {e}")
            return None


In [9]:
import os
import torch
from tqdm.auto import tqdm
import gc

# where to cache each precomputed sample
TRAIN_CACHE_DIR = "train_cache"
os.makedirs(TRAIN_CACHE_DIR, exist_ok=True)

train_ds = MaskSequenceDataset(train_triplets, NUM_CLASSES)

# find already done indices by looking at existing files
existing_files = [f for f in os.listdir(TRAIN_CACHE_DIR) if f.endswith(".pt")]
done_idxs = sorted(int(f.split(".")[0]) for f in existing_files) if existing_files else []

if done_idxs:
    start_idx = max(done_idxs) + 1
    print(f"Resuming TRAIN from idx {start_idx} (already have {len(done_idxs)} cached samples)")
else:
    start_idx = 0
    print("Starting TRAIN preprocessing from scratch")

for idx in tqdm(range(start_idx, len(train_ds)), desc="Precomputing TRAIN"):
    item = train_ds[idx]
    if item is None:
        continue

    # save this (x_seq, y) pair to disk immediately
    out_path = os.path.join(TRAIN_CACHE_DIR, f"{idx:06d}.pt")
    torch.save(item, out_path)

    # keep memory under control
    if (idx + 1) % 500 == 0:
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

print("Done precomputing TRAIN ✅")


Resuming TRAIN from idx 53120 (already have 53120 cached samples)


Precomputing TRAIN:   0%|          | 0/106566 [00:00<?, ?it/s]


RuntimeError: [enforce fail at inline_container.cc:603] . unexpected pos 576 vs 470

In [17]:
import os
import torch
from torch.utils.data import Dataset, DataLoader

class CachedFilesDataset(Dataset):
    def __init__(self, cache_dir: str):
        self.cache_dir = cache_dir
        self.files = sorted(f for f in os.listdir(cache_dir) if f.endswith(".pt"))

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        path = os.path.join(self.cache_dir, self.files[idx])
        try:
            x_seq, y = torch.load(path)
            return x_seq, y
        except Exception as e:
            # Corrupted / empty file → skip this sample
            print(f"[SKIP CORRUPT] {path}: {e}")
            return None



In [18]:
def drop_none_collate(batch):
    # remove None entries (corrupted samples)
    batch = [b for b in batch if b is not None]
    if len(batch) == 0:
        return None  # whole batch was bad

    xs, ys = zip(*batch)      # tuples of tensors
    xs = torch.stack(xs, dim=0)
    ys = torch.stack(ys, dim=0)
    return xs, ys


In [20]:
BATCH_SIZE = 4  # or your usual batch size

# If you already have train_dataset / val_dataset from before, reuse them.
# Otherwise, build them from your cache folder, e.g. "train_cache", "val_cache":

if "train_dataset" not in globals():
    train_dataset = CachedFilesDataset("train_cache")  # change name if needed

if "val_dataset" not in globals():
    val_dataset = CachedFilesDataset("val_cache")      # change name if needed

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    pin_memory=True,
    collate_fn=drop_none_collate,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=True,
    collate_fn=drop_none_collate,
)

print("DataLoaders rebuilt with safe collate_fn ✅")


DataLoaders rebuilt with safe collate_fn ✅


In [13]:
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    pin_memory=True,
)

if val_dataset is not None:
    val_loader = DataLoader(
        val_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=0,
        pin_memory=True,
    )
    print("✅ Built train_loader + val_loader")
else:
    val_loader = None
    print("✅ Built train_loader (no validation dataset found)")


✅ Built train_loader (no validation dataset found)


In [ ]:
import torch
from torch.utils.data import DataLoader

# load precomputed items (no segformer here)
train_items = torch.load("train_items.pt")
val_items   = torch.load("val_items.pt")

print(f"Loaded TRAIN items: {len(train_items)}")
print(f"Loaded VAL items:   {len(val_items)}")

train_loader = DataLoader(
    train_items,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    pin_memory=True,
)

val_loader = DataLoader(
    val_items,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=True,
)

print("DataLoaders ready ")


In [21]:
class ConvLSTMCell(nn.Module):
    def __init__(self, in_ch, hid_ch, k=3):
        super().__init__()
        p = k // 2
        self.hid_ch = hid_ch
        self.conv = nn.Conv2d(in_ch + hid_ch, 4 * hid_ch, kernel_size=k, padding=p)

    def forward(self, x, h, c):
        # x: (B, in_ch, H, W)
        # h,c: (B, hid_ch, H, W)
        combined = torch.cat([x, h], dim=1)
        gates = self.conv(combined)
        i, f, o, g = torch.chunk(gates, 4, dim=1)
        i = torch.sigmoid(i)
        f = torch.sigmoid(f)
        o = torch.sigmoid(o)
        g = torch.tanh(g)

        c_next = f * c + i * g
        h_next = o * torch.tanh(c_next)
        return h_next, c_next

class MaskForecaster(nn.Module):
    """
    Input:  (B, T, C, H, W) one-hot masks for past timesteps
    Output: (B, K, H, W) logits for next mask
    """
    def __init__(self, num_classes, hidden=64):
        super().__init__()
        self.cell = ConvLSTMCell(in_ch=num_classes, hid_ch=hidden, k=3)
        self.head = nn.Sequential(
            nn.Conv2d(hidden, hidden, 3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(hidden, num_classes, 1)
        )

    def forward(self, x_seq):
        B, T, C, H, W = x_seq.shape
        h = torch.zeros((B, self.cell.hid_ch, H, W), device=x_seq.device)
        c = torch.zeros((B, self.cell.hid_ch, H, W), device=x_seq.device)

        for t in range(T):
            x = x_seq[:, t]
            h, c = self.cell(x, h, c)

        logits = self.head(h)
        return logits

model = MaskForecaster(num_classes=NUM_CLASSES, hidden=96).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
criterion = nn.CrossEntropyLoss()


In [22]:
from tqdm import tqdm
import torch

def run_epoch(loader, train: bool):
    model.train(train)
    total_loss = 0.0
    n = 0

    pbar = tqdm(loader, leave=False)
    pbar.set_description("Train" if train else "Val")

    for x_seq, y in pbar:
        x_seq = x_seq.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)

        logits = model(x_seq)
        loss = criterion(logits, y)

        if train:
            optimizer.zero_grad(set_to_none=True)
            loss.backward()
            optimizer.step()

        bs = x_seq.size(0)
        total_loss += loss.item() * bs
        n += bs

        pbar.set_postfix(loss=f"{loss.item():.4f}")

    return total_loss / max(1, n)


In [23]:
from tqdm import trange
import time

patience = 10
min_delta = 1e-4
wait = 0

best_val = float("inf")

epoch_bar = trange(1, EPOCHS+1)

start_time = time.time()

for epoch in epoch_bar:
    train_loss = run_epoch(train_loader, train=True)
    val_loss   = run_epoch(val_loader, train=False)

    epoch_bar.set_description(f"Epoch {epoch}")
    epoch_bar.set_postfix(train=f"{train_loss:.4f}", val=f"{val_loss:.4f}")

    # --- early stopping & saving ---
    if val_loss < best_val - min_delta:
        best_val = val_loss
        wait = 0
        torch.save(model.state_dict(), "mask_forecaster_best.pt")
    else:
        wait += 1

    if wait >= patience:
        print("Early stopping triggered.")
        break


  0%|          | 0/25 [00:00<?, ?it/s]C:\Users\RA-RV\AppData\Local\Temp\ipykernel_37844\580937558.py:16: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  x_seq, y = torch.load(

[SKIP CORRUPT] train_cache\053118.pt: PytorchStreamReader failed reading zip archive: failed finding central directory


  0%|          | 0/25 [15:50<?, ?it/s]


KeyboardInterrupt: 

In [ ]:
# Load best model
model.load_state_dict(torch.load("mask_forecaster_best.pt", map_location=device))
model.eval()

@torch.no_grad()
def predict_next_mask_from_two_images(img_prev: Image.Image, img_curr: Image.Image) -> np.ndarray:
    """
    Predict next mask given two images (prev, curr) using:
      SegFormer -> masks -> ConvLSTM forecaster -> next mask
    Returns: (H, W) numpy uint8 of class ids
    """
    m_prev = segformer_predict_mask(img_prev)   # (H,W)
    m_curr = segformer_predict_mask(img_curr)

    oh_prev = F.one_hot(m_prev, num_classes=NUM_CLASSES).permute(2,0,1).float()
    oh_curr = F.one_hot(m_curr, num_classes=NUM_CLASSES).permute(2,0,1).float()

    x_seq = torch.stack([oh_prev, oh_curr], dim=0).unsqueeze(0).to(device)  # (1,2,C,H,W)
    logits = model(x_seq)                                                   # (1,K,H,W)
    pred = torch.argmax(logits, dim=1).squeeze(0).to("cpu").numpy().astype(np.uint8)
    return pred

# Example: pick one tile from val_triplets
p2014, p2020, p2025, tid = val_triplets[0]
img2020 = Image.open(p2020).convert("RGB")
img2025 = Image.open(p2025).convert("RGB")

mask2030 = predict_next_mask_from_two_images(img2020, img2025)

print("Predicted 2030 mask shape:", mask2030.shape, "unique classes:", np.unique(mask2030))
